# Tavily + Oracle AI Database memory demo

This notebook shows Tavily as the freshness layer and Oracle as the memory layer: Tavily search, Oracle write-through persistence, freshness-cache hit behavior, provenance inspection, and semantic deduplication.

In [ ]:
import os
import oracledb
from tavily import TavilyHybridClient

connection = oracledb.connect(
    user=os.environ["ORACLE_USER"],
    password=os.environ["ORACLE_PASSWORD"],
    dsn=os.environ.get("ORACLE_DSN", "localhost:1521/FREE"),
)

common_config = {
    "api_key": os.environ["TAVILY_API_KEY"],
    "db_provider": "oracle",
    "connection": connection,
    "table_name": os.environ.get("ORACLE_VECTOR_TABLE", "TAVILY_DOCUMENTS"),
    "embeddings_field": os.environ.get("ORACLE_EMBEDDINGS_FIELD", "EMBEDDINGS"),
    "content_field": os.environ.get("ORACLE_CONTENT_FIELD", "CONTENT"),
}

In [ ]:
client = TavilyHybridClient(
    **common_config,
    retrieval_mode="freshness_cache",
    cache_ttl_seconds=3600,
    cache_score_threshold=0.75,
    enable_native_hybrid_search=True,
    enable_oracle_json_payload=True,
    enable_provenance_metadata=True,
    dedup_similarity_threshold=0.95,
)

tavily_preview = client.tavily.search(
    "latest Oracle Database vector search features",
    max_results=3,
)
tavily_preview["results"]

In [ ]:
results = client.search(
    "latest Oracle Database vector search features",
    max_results=5,
    max_local=5,
    max_foreign=5,
    save_foreign=True,
)
results

In [ ]:
cache_hit_results = client.search(
    "latest Oracle Database vector search features",
    max_results=5,
    max_local=5,
    max_foreign=5,
    save_foreign=True,
)
cache_hit_results

In [ ]:
with connection.cursor() as cursor:
    cursor.execute(
        f"""
        SELECT SOURCE_URL,
               RETRIEVAL_QUERY,
               RETRIEVAL_MODE,
               INSERTED_FROM,
               JSON_VALUE(RAW_PAYLOAD, '$.provenance.provider_name') AS PROVIDER
        FROM {common_config['table_name']}
        WHERE JSON_EXISTS(RAW_PAYLOAD, '$.provenance')
        FETCH FIRST 5 ROWS ONLY
        """
    )
    provenance_rows = cursor.fetchall()

provenance_rows

In [ ]:
dedup_results = client.search(
    "latest Oracle Database vector search features",
    max_results=5,
    max_local=0,
    max_foreign=5,
    save_foreign=True,
)
dedup_results